# Regionalización hidroclimática de estaciones meteorológicas

## Código para la reproducción de resultados y figuras

El presente cuaderno contiene los procedimientos computacionales empleados para la regionalización hidroclimática de las estaciones meteorológicas consideradas en el estudio.

El análisis comprende la lectura y preparación de la base de datos, el cálculo de la climatología mensual de precipitación, la obtención de indicadores de estacionalidad, la estandarización de las variables utilizadas en el agrupamiento y la aplicación del método jerárquico de Ward mediante distancia euclidiana.

Asimismo, se incluye la evaluación del número de conglomerados mediante el estadístico Gap y la representación gráfica de los resultados obtenidos.

La base de datos utilizada corresponde a:

`01_datos/raw/Base_Datos_Final_ETo_Pivot.csv`

La solución final de regionalización considera **tres conglomerados hidroclimáticos (k = 3)**, de acuerdo con la clasificación utilizada en los análisis y resultados finales de la investigación.

El cuaderno permite reproducir las siguientes representaciones gráficas:

1. Climatología mensual de precipitación por estación.
2. Dendrograma del agrupamiento jerárquico mediante el método de Ward.
3. Selección del número de conglomerados mediante el estadístico Gap.
4. Distribución espacial de los conglomerados de estaciones meteorológicas.
5. Visualización multivariada de los conglomerados obtenidos.

## 1. Paquetes


In [ ]:
if (!require("pacman")) install.packages("pacman")  # Instala pacman si no está disponible.

pacman::p_load(

  tidyverse,   # Manipulación, transformación y visualización de datos.
  lubridate,   # Manejo de fechas y componentes temporales.
  janitor,     # Limpieza y estandarización de nombres de variables.
  skimr,       # Resumen descriptivo y exploración de datos.
  naniar,      # Identificación y análisis de datos faltantes.
  cluster,     # Métodos de análisis de conglomerados.
  factoextra,  # Visualización y evaluación de resultados de clustering.
  maps,        # Datos cartográficos básicos para representación espacial.
  ggrepel      # Etiquetas de texto evitando superposición en gráficos.

)

## 2. Rutas del proyecto


In [ ]:
# ============================================================
# RUTAS DEL PROYECTO - WINDOWS
# ============================================================

encontrar_raiz_proyecto <- function(inicio = getwd()) {
  marcadores <- c("01_datos", "02_scripts", "03_resultados", "04_documentacion")
  es_raiz <- function(x) all(dir.exists(file.path(x, marcadores)))

  raiz_env <- Sys.getenv("TESIS_SEQUIAS_ROOT", unset = "")
  if (nzchar(raiz_env)) {
    raiz_env <- normalizePath(raiz_env, winslash = "/", mustWork = FALSE)
    if (es_raiz(raiz_env)) return(raiz_env)
  }

  ruta <- normalizePath(inicio, winslash = "/", mustWork = TRUE)

  repeat {
    if (es_raiz(ruta)) return(ruta)

    padre <- dirname(ruta)
    if (identical(padre, ruta)) {
      stop(
        "No se encontró la raíz del proyecto. ",
        "Ejecute Jupyter dentro del repositorio o defina TESIS_SEQUIAS_ROOT."
      )
    }

    ruta <- padre
  }
}

ruta_base <- encontrar_raiz_proyecto()

ruta_raw        <- file.path(ruta_base, "01_datos", "raw")          # Datos originales.
ruta_inter      <- file.path(ruta_base, "01_datos", "intermedios")  # Datos intermedios.
ruta_procesados <- file.path(ruta_base, "01_datos", "procesados")   # Datos procesados.
ruta_metadata   <- file.path(ruta_base, "01_datos", "metadata")     # Metadatos del estudio.

ruta_scripts <- file.path(
  ruta_base, "02_scripts", "01_regionalizacion"
)  # Código del análisis de regionalización.

ruta_tablas <- file.path(
  ruta_base, "03_resultados", "01_regionalizacion", "tablas"
)  # Tablas generadas por el análisis.

ruta_figuras <- file.path(
  ruta_base, "03_resultados", "01_regionalizacion", "figuras"
)  # Figuras generadas por el análisis.

ruta_mapas <- ruta_figuras  # Mapas almacenados junto con las figuras.
ruta_csv   <- ruta_tablas    # Archivos CSV almacenados junto con las tablas.

archivo_base <- file.path(
  ruta_raw, "Base_Datos_Final_ETo_Pivot.csv"
)  # Base de datos utilizada en la regionalización.

cat("Archivo base:\n", archivo_base)  # Muestra la ruta del archivo de entrada.
cat("\n¿Existe el archivo?: ", file.exists(archivo_base), "\n")  # Verifica su existencia.


## 3. Lectura de la base


In [ ]:
df <- read.csv(archivo_base, stringsAsFactors = FALSE)  # Importa la base de datos en formato CSV.

df <- df %>%                                      # Inicia la transformación de la base.
  clean_names() %>%                               # Estandariza los nombres de las variables.
  mutate(                                         # Crea variables temporales adicionales.
    fecha = as.Date(fecha),                       # Convierte "fecha" al formato de fecha de R.
    anio  = year(fecha),                          # Extrae el año de cada registro.
    mes   = month(fecha)                          # Extrae el mes de cada registro.
  )


## 4. Climatología mensual


In [ ]:
# CLIMATOLOGÍA MENSUAL

clim_mensual <- df %>%                                      # Inicia el cálculo de la climatología mensual.
  group_by(estacion, latitud, longitud, altitud, mes) %>%   # Agrupa los registros por estación y mes.
  summarise(
    ppt_media_mensual = mean(pt101, na.rm = TRUE),          # Calcula la precipitación media mensual.
    .groups = "drop"                                        # Finaliza el agrupamiento después del cálculo.
  )


## Figura 1. Climatología mensual de precipitación por estación


In [ ]:
# REPRESENTACIÓN DE LA CLIMATOLOGÍA MENSUAL

ggplot(clim_mensual, aes(x = mes, y = ppt_media_mensual, group = estacion)) +  # Define meses, precipitación y estaciones.
  geom_line(alpha = 0.4) +                                                     # Representa el régimen mensual mediante líneas.
  geom_point(size = 1) +                                                       # Señala los valores medios de cada mes.
  scale_x_continuous(breaks = 1:12) +                                          # Muestra los 12 meses en el eje horizontal.
  theme_minimal() +                                                            # Aplica un formato gráfico simplificado.
  labs(
    title = "Climatología mensual de precipitación por estación",              # Título de la figura.
    x = "Mes",                                                                 # Etiqueta del eje horizontal.
    y = "Precipitación media mensual (mm)"                                     # Etiqueta y unidad del eje vertical.
  )

ggsave(
  filename = file.path(ruta_figuras, "climatologia_mensual_estaciones.png"),   # Define el archivo de salida.
  width = 10,                                                                  # Establece el ancho de la figura.
  height = 6,                                                                  # Establece la altura de la figura.
  dpi = 300                                                                    # Guarda la figura a 300 dpi.
)


## 5. Preparación de la base para clustering


In [ ]:
# ORGANIZACIÓN DE LA CLIMATOLOGÍA MENSUAL PARA EL ANÁLISIS DE CLUSTERS

clim_wide <- clim_mensual %>%                                                    # Parte de la climatología mensual por estación.
  mutate(mes_lab = sprintf("p_%02d", mes)) %>%                                   # Crea etiquetas mensuales p_01, ..., p_12.
  select(estacion, latitud, longitud, altitud, mes_lab, ppt_media_mensual) %>%   # Selecciona variables requeridas.
  pivot_wider(
    names_from = mes_lab,                                                        # Convierte cada mes en una columna.
    values_from = ppt_media_mensual                                              # Asigna la precipitación media mensual como valor.
  )


In [ ]:
# CÁLCULO DE INDICADORES DE ESTACIONALIDAD DE LA PRECIPITACIÓN

indices_estacionalidad <- clim_mensual %>%                                      # Utiliza la climatología mensual.
  group_by(estacion) %>%                                                        # Calcula los indicadores por estación.
  summarise(
    ppt_anual = sum(ppt_media_mensual, na.rm = TRUE),                           # Precipitación media anual acumulada.
    ppt_djfm = sum(ppt_media_mensual[mes %in% c(12, 1, 2, 3)], na.rm = TRUE),   # Precipitación de diciembre-marzo.
    ppt_mjjas = sum(ppt_media_mensual[mes %in% c(5, 6, 7, 8, 9)], na.rm = TRUE),# Precipitación de mayo-septiembre.
    cv_mensual = sd(ppt_media_mensual, na.rm = TRUE) /
                 mean(ppt_media_mensual, na.rm = TRUE),                         # Variabilidad relativa mensual (CV).
    mes_pico = mes[which.max(ppt_media_mensual)][1],                            # Mes de máxima precipitación media.
    .groups = "drop"                                                            # Finaliza el agrupamiento.
  ) %>%
  mutate(
    frac_djfm = ppt_djfm / ppt_anual,                                           # Fracción anual concentrada en diciembre-marzo.
    frac_mjjas = ppt_mjjas / ppt_anual                                          # Fracción anual concentrada en mayo-septiembre.
  )

In [ ]:
# INTEGRACIÓN DE VARIABLES PARA EL ANÁLISIS DE CLUSTERS

base_cluster <- clim_wide %>%                                      # Parte de las variables climatológicas mensuales.
  left_join(indices_estacionalidad, by = "estacion")               # Incorpora los indicadores de estacionalidad por estación.


In [ ]:
# SELECCIÓN DE VARIABLES PARA EL ANÁLISIS DE CLUSTERS

vars_cluster <- base_cluster %>%                  # Parte de la base integrada por estación.
  select(
    starts_with("p_"),                            # Precipitación mensual: p_01 a p_12.
    altitud,                                      # Altitud de la estación (m s. n. m.).
    ppt_anual,                                    # Precipitación media anual acumulada.
    frac_djfm,                                    # Fracción de precipitación de diciembre-marzo.
    cv_mensual                                    # Coeficiente de variación mensual.
  )

rownames(vars_cluster) <- base_cluster$estacion   # Asigna cada estación como identificador de fila.

In [ ]:
mat_cluster <- scale(vars_cluster)


## Figura 2. Dendrograma — Ward.D2


In [ ]:
dist_mat <- dist(mat_cluster, method = "euclidean")
hc_ward <- hclust(dist_mat, method = "ward.D2")

plot(hc_ward, cex = 0.8, main = "Dendrograma - Clustering jerárquico (Ward)")


## Figura 3. Selección del número de clusters — Gap Statistic


In [ ]:
# DETERMINACIÓN DEL NÚMERO ÓPTIMO DE CLUSTERS MEDIANTE GAP STATISTIC

fviz_nbclust(
  mat_cluster,                       # Matriz de variables hidroclimáticas estandarizadas.
  FUN = hcut,                        # Aplica clustering jerárquico para evaluar distintos valores de k.
  method = "gap_stat"                # Determina el número de clusters mediante Gap Statistic.
) +
  labs(
    title = "Selección del número de clusters - Gap Statistic"  # Título de la figura.
  )


## 6. Asignación final de clusters


In [ ]:
k <- 3
base_cluster$cluster_ward <- factor(cutree(hc_ward, k = k))


## Figura 4. Distribución espacial de los clusters


In [ ]:
mapa_mundo <- map_data("world")

ggplot() +
  geom_polygon(
    data = mapa_mundo,
    aes(x = long, y = lat, group = group),
    fill = "gray95", color = "gray70", linewidth = 0.2
  ) +
  geom_point(
    data = base_cluster,
    aes(x = longitud, y = latitud, color = cluster_ward),
    size = 3
  ) +
  geom_text_repel(
    data = base_cluster,
    aes(x = longitud, y = latitud, label = estacion, color = cluster_ward),
    size = 3,
    max.overlaps = 30
  ) +
  coord_fixed(xlim = c(-76, -68), ylim = c(-19.5, -12)) +
  theme_minimal() +
  labs(
    title = "Clusters de estaciones según régimen pluviométrico",
    x = "Longitud",
    y = "Latitud",
    color = "Cluster"
  )

ggsave(
  file.path(ruta_mapas, "mapa_clusters_ward.png"),
  width = 10,
  height = 8,
  dpi = 300
)


## Figura 5. Visualización multivariada de los clusters


In [ ]:
fviz_cluster(
  list(data = mat_cluster, cluster = base_cluster$cluster_ward),
  geom = "point",
  ellipse.type = "convex",
  main = "Visualización de clusters"
)
